# Chapter 19: Deployment — Putting Models Into Production

A model in a notebook is a toy. A model serving real users is a **product**.

```
Research:    train → evaluate → publish paper
Production:  train → package → deploy → monitor → update → repeat
```

This chapter covers how models go from experiments to serving millions of requests.

---
## The Deployment Pipeline

```
┌─────────┐   ┌─────────┐   ┌─────────┐   ┌─────────┐   ┌─────────┐
│  Train  │ → │ Package │ → │  Deploy │ → │  Serve  │ → │ Monitor │
│         │   │         │   │         │   │         │   │         │
│ model   │   │ Docker  │   │ cloud/  │   │ API     │   │ latency │
│ weights │   │ + deps  │   │ edge    │   │ requests│   │ errors  │
└─────────┘   └─────────┘   └─────────┘   └─────────┘   └─────────┘
```

In [ ]:
import numpy as np
np.random.seed(42)
import time

# Deployment options overview
print("Deployment Options for ML Models:\n")

options = [
    ("API Service",      "Cloud GPU",    "10-500ms",  "High",    "ChatGPT, Claude API"),
    ("Serverless",       "Cloud (auto)", "100-2000ms","Medium",  "AWS Lambda + model"),
    ("Edge/On-device",   "Phone/laptop", "5-50ms",    "Low",     "Siri, autocorrect"),
    ("Batch processing", "Cloud CPU/GPU","Minutes",   "Lowest",  "Recommendation refresh"),
    ("Embedded",         "IoT/car",      "1-10ms",    "Lowest",  "Self-driving, sensors"),
]

print(f"  {'Option':<18} {'Hardware':<14} {'Latency':<12} {'Cost':<10} {'Example'}")
print(f"  {'─'*18} {'─'*14} {'─'*12} {'─'*10} {'─'*25}")
for opt, hw, lat, cost, ex in options:
    print(f"  {opt:<18} {hw:<14} {lat:<12} {cost:<10} {ex}")

print(f"\n  Choice depends on:")
print(f"    • Latency requirement (real-time chat vs batch analytics)")
print(f"    • Model size (7B params won't run on a phone)")
print(f"    • Cost budget (GPU servers are expensive)")
print(f"    • Privacy (some data can't leave the device)")

---
## Serving Models: The API Pattern

Most production ML models are served as **HTTP APIs**:

```
Client → POST /predict {"text": "Hello"} → Server → Model → Response
```

The model lives on a server with GPUs, clients send requests over the network.

In [ ]:
# Simulated model server
print("Model Serving: API Pattern\n")

# Simulate a simple model server
class ModelServer:
    def __init__(self, model_name, model_size_gb):
        self.model_name = model_name
        self.model_size = model_size_gb
        self.loaded = False
        self.request_count = 0
    
    def load_model(self):
        """Load model into GPU memory (done once at startup)."""
        self.loaded = True
        return f"Loaded {self.model_name} ({self.model_size}GB) into GPU memory"
    
    def predict(self, input_text):
        """Handle a single prediction request."""
        if not self.loaded:
            return {"error": "Model not loaded"}
        
        self.request_count += 1
        # Simulate inference time
        latency_ms = np.random.exponential(50)  # ~50ms average
        
        return {
            "model": self.model_name,
            "input": input_text,
            "output": f"[generated response for '{input_text[:30]}...']",
            "latency_ms": round(latency_ms, 1),
            "request_id": self.request_count,
        }

# Start server
server = ModelServer("llama-7b", model_size_gb=14)
print(f"  {server.load_model()}\n")

# Handle requests
requests = [
    "What is the capital of France?",
    "Explain quantum computing in simple terms.",
    "Write a Python function to sort a list.",
    "Translate 'hello' to Spanish.",
    "Summarize this article about climate change.",
]

print(f"  Handling {len(requests)} requests:\n")
print(f"  {'#':<4} {'Latency':>8} {'Input (truncated)'}")
print(f"  {'─'*4} {'─'*8} {'─'*40}")

total_latency = 0
for req in requests:
    result = server.predict(req)
    total_latency += result['latency_ms']
    print(f"  {result['request_id']:<4} {result['latency_ms']:>6.1f}ms '{req[:40]}'")

print(f"\n  Average latency: {total_latency/len(requests):.1f}ms")
print(f"  Throughput: ~{1000/(total_latency/len(requests)):.0f} requests/sec (single GPU)")

In [ ]:
# Real-world API structure
print("Real-World Model API (what it looks like):\n")

print("  REQUEST:")
print("  ┌─────────────────────────────────────────────────────────┐")
print("  │ POST /v1/chat/completions                              │")
print("  │ Authorization: Bearer sk-...                           │")
print("  │                                                         │")
print("  │ {                                                       │")
print('  │   "model": "claude-sonnet-4-20250514",                        │')
print('  │   "messages": [                                        │')
print('  │     {"role": "user", "content": "Explain RAG"}         │')
print("  │   ],                                                    │")
print('  │   "max_tokens": 500,                                   │')
print('  │   "temperature": 0.7                                   │')
print("  │ }                                                       │")
print("  └─────────────────────────────────────────────────────────┘")

print("\n  RESPONSE:")
print("  ┌─────────────────────────────────────────────────────────┐")
print("  │ {                                                       │")
print('  │   "id": "msg_01XYZ...",                                │')
print('  │   "model": "claude-sonnet-4-20250514",                        │')
print('  │   "content": "RAG is a technique that...",             │')
print('  │   "usage": {                                           │')
print('  │     "input_tokens": 12,                                │')
print('  │     "output_tokens": 150                               │')
print("  │   },                                                    │")
print('  │   "latency_ms": 1200                                   │')
print("  │ }                                                       │")
print("  └─────────────────────────────────────────────────────────┘")

print("\n  This is exactly what happens when you call Claude or GPT-4.")
print("  Your app sends HTTP requests to their API servers.")

---
## Optimization: Making Models Fast Enough

LLMs are slow and expensive. Key optimization techniques:

```
Technique         Speedup    Quality Loss    Complexity
────────────────────────────────────────────────────────
Quantization      2-4x       Minimal         Low
KV-Cache          2-10x      None            Medium
Batching          5-20x      None            Medium
Speculative       2-3x       None            High
Distillation      10-100x    Moderate        High
```

In [ ]:
# Optimization techniques explained
print("Model Optimization Techniques:\n")

print("  1. QUANTIZATION (Ch 15 recap)")
print("     Reduce precision: FP16 → INT8 → INT4")
print("     FP16: 14GB → INT4: 3.5GB (for 7B model)")
print("     Speedup: 2-4x (less memory bandwidth needed)\n")

print("  2. KV-CACHE (Key-Value Cache)")
print("     Problem: generating token N recomputes attention for tokens 1..N-1")
print("     Solution: cache K and V matrices from previous tokens")
print("     ┌─────────────────────────────────────────────┐")
print("     │ Without cache: generate 100 tokens          │")
print("     │   Token 1:   compute attention over 1 token │")
print("     │   Token 2:   recompute over 2 tokens        │")
print("     │   Token 100: recompute over 100 tokens      │")
print("     │   Total: 1+2+3+...+100 = 5050 operations   │")
print("     │                                              │")
print("     │ With cache: generate 100 tokens              │")
print("     │   Token 1:   compute and CACHE               │")
print("     │   Token 2:   only new token + read cache     │")
print("     │   Token 100: only new token + read cache     │")
print("     │   Total: 100 operations (50x fewer!)         │")
print("     └─────────────────────────────────────────────┘\n")

print("  3. BATCHING")
print("     Process multiple requests simultaneously on the GPU.")
print("     GPU is massively parallel — 1 request uses 5% of capacity!")
print("     Batch of 32: nearly same latency, 32x throughput.\n")

print("  4. SPECULATIVE DECODING")
print("     Use a small fast model to 'draft' tokens,")
print("     then verify them in parallel with the big model.")
print("     Small model: 1B params (fast, often correct)")
print("     Big model: 70B params (slow, verify draft in one pass)")
print("     If draft is correct: saved time! If wrong: discard and redo.")

In [ ]:
# Batching simulation
print("Batching: Process Multiple Requests Together\n")

# Simulate GPU utilization
def simulate_serving(n_requests, batch_size, per_token_ms=5, tokens_per_request=50):
    total_time = 0
    n_batches = (n_requests + batch_size - 1) // batch_size
    
    for _ in range(n_batches):
        # Batch processing: all items processed in parallel
        # Latency slightly increases with batch size (memory bandwidth)
        batch_latency = per_token_ms * tokens_per_request * (1 + 0.1 * batch_size)
        total_time += batch_latency
    
    throughput = n_requests / (total_time / 1000)  # requests per second
    avg_latency = total_time / n_batches
    return throughput, avg_latency

n_requests = 64
print(f"  Serving {n_requests} requests with different batch sizes:\n")
print(f"  {'Batch Size':<12} {'Throughput':>12} {'Avg Latency':>12} {'GPU Util':>10}")
print(f"  {'─'*12} {'─'*12} {'─'*12} {'─'*10}")

for bs in [1, 4, 8, 16, 32]:
    tp, lat = simulate_serving(n_requests, bs)
    gpu_util = min(100, bs * 5)  # rough approximation
    print(f"  {bs:<12} {tp:>9.1f}/s {lat:>10.0f}ms {gpu_util:>8}%")

print(f"\n  Key insight: batch=1 wastes most GPU capacity.")
print(f"  Larger batches = higher throughput but slightly more latency.")
print(f"\n  Continuous batching (vLLM, TGI):")
print(f"    Don't wait for full batch — add new requests as old ones finish.")
print(f"    Maximizes GPU utilization without waiting.")

---
## Scaling: Handling More Traffic

One GPU can handle ~10-100 requests/second. For millions of users:

```
Horizontal scaling: add more GPU servers behind a load balancer

         ┌── GPU Server 1 ──┐
Users → Load Balancer → GPU Server 2
         └── GPU Server 3 ──┘
```

In [ ]:
# Scaling simulation
print("Scaling: From 1 GPU to a Fleet\n")

# Cost model for GPU serving
gpu_cost_per_hour = 3.50  # A100 80GB on cloud
requests_per_gpu_per_second = 50  # with batching

traffic_scenarios = [
    ("Startup (early)",    10,        "1 request every 6 seconds"),
    ("Growing app",        100,       "a few concurrent users"),
    ("Popular product",    1000,      "hundreds of concurrent users"),
    ("Large scale",        10000,     "thousands of concurrent users"),
    ("ChatGPT-scale",      100000,    "millions of users"),
]

print(f"  {'Scenario':<20} {'Req/min':>8} {'GPUs':>5} {'Cost/hr':>9} {'Cost/month':>11}")
print(f"  {'─'*20} {'─'*8} {'─'*5} {'─'*9} {'─'*11}")

for name, rpm, desc in traffic_scenarios:
    rps = rpm / 60
    gpus_needed = max(1, int(np.ceil(rps / requests_per_gpu_per_second)))
    cost_hr = gpus_needed * gpu_cost_per_hour
    cost_month = cost_hr * 24 * 30
    
    if cost_month > 1e6:
        cost_str = f"${cost_month/1e6:.1f}M"
    elif cost_month > 1e3:
        cost_str = f"${cost_month/1e3:.0f}K"
    else:
        cost_str = f"${cost_month:.0f}"
    
    print(f"  {name:<20} {rpm:>8} {gpus_needed:>5} ${cost_hr:>7.0f} {cost_str:>11}")

print(f"\n  Scaling strategies:")
print(f"    • Autoscaling: add GPUs during peak, remove during quiet")
print(f"    • Caching: store common responses, skip model inference")
print(f"    • Smaller models: use 7B instead of 70B where acceptable")
print(f"    • Rate limiting: cap requests per user to control costs")

---
## Monitoring: Knowing When Things Go Wrong

In production, you need to track:
- **Latency**: how fast are responses?
- **Errors**: are requests failing?
- **Quality**: is the model output getting worse over time?
- **Cost**: are we within budget?

In [ ]:
# Production monitoring simulation
print("Production Monitoring Dashboard:\n")

# Simulate 24 hours of traffic
np.random.seed(42)
hours = 24

# Traffic pattern (more during day)
traffic_pattern = np.array([0.2, 0.1, 0.1, 0.1, 0.1, 0.2, 0.4, 0.7,
                           0.9, 1.0, 1.0, 0.9, 0.8, 0.9, 1.0, 0.9,
                           0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.3, 0.2])

base_rps = 100
requests_per_hour = (traffic_pattern * base_rps * 3600).astype(int)

# Metrics
latencies = 50 + 20 * traffic_pattern + np.random.randn(24) * 5
error_rates = 0.001 + 0.002 * traffic_pattern + np.maximum(0, np.random.randn(24) * 0.001)
# Simulate a spike at hour 14 (incident)
latencies[14] = 250
error_rates[14] = 0.05

print(f"  Hour  Requests  Latency(ms)  Error Rate  Status")
print(f"  {'─'*4}  {'─'*8}  {'─'*11}  {'─'*10}  {'─'*10}")

for h in range(24):
    status = "OK"
    if latencies[h] > 200:
        status = "⚠ HIGH LAT"
    if error_rates[h] > 0.01:
        status = "🔴 ALERT"
    
    req_str = f"{requests_per_hour[h]:,}"
    print(f"  {h:>2}:00  {req_str:>8}  {latencies[h]:>9.0f}ms  {error_rates[h]:>9.3%}  {status}")

print(f"\n  Hour 14: latency spike + error surge = model server overloaded!")
print(f"  Action: autoscaler added 2 GPUs, recovered in 15 minutes.")

In [ ]:
# Model drift detection
print("Model Drift: When Production Data Changes\n")

print("  The model was trained on data from 2023.")
print("  But the world changes! New slang, new topics, new patterns.\n")

print("  Types of drift:\n")
print("  1. DATA DRIFT (input distribution changes)")
print("     Training: mostly English text")
print("     Production: suddenly 30% Spanish (new market launch)")
print("     → Model accuracy drops on Spanish inputs\n")

print("  2. CONCEPT DRIFT (relationship between input→output changes)")
print("     Training: 'sick' = ill")
print("     Production: 'sick' = awesome (slang shift)")
print("     → Model misinterprets sentiment\n")

print("  3. LABEL DRIFT (output distribution changes)")
print("     Training: 5% of emails are spam")
print("     Production: 20% are spam (new spam campaign)")
print("     → Model's threshold is miscalibrated\n")

print("  Detection:")
print("    • Monitor input statistics (token distributions, input lengths)")
print("    • Track prediction confidence over time")
print("    • Sample outputs for human review weekly")
print("    • A/B test new model versions against production")

print("\n  Response:")
print("    • Retrain on recent data")
print("    • Update RAG knowledge base")
print("    • Adjust decision thresholds")
print("    • Roll back if new model is worse")

---
## A/B Testing: Comparing Model Versions

Before replacing your production model, test the new one on real traffic:

```
90% of users → Current model (v1)
10% of users → New model (v2)

Compare metrics after 1 week → promote or rollback
```

In [ ]:
# A/B testing simulation
print("A/B Testing Model Versions:\n")

np.random.seed(42)
n_requests = 10000

# Model A (current): good but slightly slower
# Model B (new): faster, maybe slightly better quality

# Assign traffic: 90% A, 10% B
assignment = np.random.choice(['A', 'B'], size=n_requests, p=[0.9, 0.1])

# Simulate metrics
latency_a = np.random.exponential(100, size=np.sum(assignment == 'A'))  # 100ms avg
latency_b = np.random.exponential(70, size=np.sum(assignment == 'B'))   # 70ms avg

# Quality: user satisfaction (1-5 rating)
quality_a = np.clip(np.random.normal(4.0, 0.5, size=np.sum(assignment == 'A')), 1, 5)
quality_b = np.clip(np.random.normal(4.1, 0.5, size=np.sum(assignment == 'B')), 1, 5)

# Error rates
errors_a = np.random.binomial(1, 0.02, size=np.sum(assignment == 'A'))
errors_b = np.random.binomial(1, 0.015, size=np.sum(assignment == 'B'))

print(f"  Traffic split: A={np.sum(assignment=='A')} requests, B={np.sum(assignment=='B')} requests\n")

print(f"  {'Metric':<20} {'Model A (current)':>18} {'Model B (new)':>15} {'Winner':>8}")
print(f"  {'─'*20} {'─'*18} {'─'*15} {'─'*8}")

metrics = [
    ("Avg latency", f"{latency_a.mean():.0f}ms", f"{latency_b.mean():.0f}ms", "B" if latency_b.mean() < latency_a.mean() else "A"),
    ("P95 latency", f"{np.percentile(latency_a, 95):.0f}ms", f"{np.percentile(latency_b, 95):.0f}ms", "B" if np.percentile(latency_b, 95) < np.percentile(latency_a, 95) else "A"),
    ("Avg quality", f"{quality_a.mean():.3f}", f"{quality_b.mean():.3f}", "B" if quality_b.mean() > quality_a.mean() else "A"),
    ("Error rate", f"{errors_a.mean():.3%}", f"{errors_b.mean():.3%}", "B" if errors_b.mean() < errors_a.mean() else "A"),
]

for name, a_val, b_val, winner in metrics:
    print(f"  {name:<20} {a_val:>18} {b_val:>15} {winner:>8}")

print(f"\n  Decision: Model B wins on all metrics → promote to 100% traffic")
print(f"\n  Rollout strategy (gradual):")
print(f"    Week 1: 10% → B  (validate, catch issues)")
print(f"    Week 2: 50% → B  (broader validation)")
print(f"    Week 3: 100% → B (full rollout)")
print(f"    Keep A ready for instant rollback if needed")

---
## Safety in Production

LLMs can generate harmful content. Production systems need guardrails:

In [ ]:
# Safety guardrails
print("Production Safety Guardrails:\n")

print("  ┌─────────────────────────────────────────────────────────────┐")
print("  │                    Request Flow                             │")
print("  │                                                             │")
print("  │  User Input                                                 │")
print("  │      ↓                                                      │")
print("  │  ┌─────────────────┐                                       │")
print("  │  │ INPUT FILTER    │ Block: jailbreaks, PII, harmful prompts│")
print("  │  └────────┬────────┘                                       │")
print("  │           ↓                                                 │")
print("  │  ┌─────────────────┐                                       │")
print("  │  │ RATE LIMITER    │ Max N requests per user per minute     │")
print("  │  └────────┬────────┘                                       │")
print("  │           ↓                                                 │")
print("  │  ┌─────────────────┐                                       │")
print("  │  │ LLM INFERENCE   │ Generate response                     │")
print("  │  └────────┬────────┘                                       │")
print("  │           ↓                                                 │")
print("  │  ┌─────────────────┐                                       │")
print("  │  │ OUTPUT FILTER   │ Block: toxic, PII, harmful content    │")
print("  │  └────────┬────────┘                                       │")
print("  │           ↓                                                 │")
print("  │  Response to User                                           │")
print("  └─────────────────────────────────────────────────────────────┘")

print(f"\n  Safety layers:")
print(f"    1. Input filtering:  detect jailbreak attempts, block harmful queries")
print(f"    2. System prompt:    instruct model on boundaries and behavior")
print(f"    3. Output filtering: scan generated text before showing to user")
print(f"    4. Rate limiting:    prevent abuse and cost explosions")
print(f"    5. Logging:          record interactions for review")
print(f"    6. Human review:     sample outputs regularly for quality")

---
## MLOps: The Full Lifecycle

**MLOps** = DevOps for ML. Managing the full model lifecycle:

```
┌──────────────────────────────────────────────┐
│              ML Lifecycle                     │
│                                              │
│  Data → Train → Evaluate → Deploy → Monitor │
│    ↑                                    │    │
│    └────────────── Retrain ←────────────┘    │
│                                              │
└──────────────────────────────────────────────┘
```

In [ ]:
# MLOps tools and practices
print("MLOps: Tools for Each Stage\n")

stages = [
    ("Data Management",   ["DVC (version data)", "Label Studio (annotate)", "Great Expectations (validate)"]),
    ("Experiment Track",  ["MLflow", "Weights & Biases (W&B)", "Neptune"]),
    ("Training",          ["PyTorch/JAX", "DeepSpeed (distributed)", "Cloud GPUs (A100, H100)"]),
    ("Model Registry",    ["MLflow", "HuggingFace Hub", "Vertex AI"]),
    ("Serving",           ["vLLM", "TGI (HuggingFace)", "Triton (NVIDIA)"]),
    ("Orchestration",     ["Kubernetes", "Docker", "Terraform"]),
    ("Monitoring",        ["Prometheus/Grafana", "Datadog", "WhyLabs"]),
    ("CI/CD",             ["GitHub Actions", "Jenkins", "Argo Workflows"]),
]

for stage, tools in stages:
    print(f"  {stage}:")
    for tool in tools:
        print(f"    • {tool}")
    print()

print(f"  Key MLOps principles:")
print(f"    • Automate everything repeatable")
print(f"    • Version data AND models AND code")
print(f"    • Test before deploy (shadow mode, A/B tests)")
print(f"    • Monitor continuously (drift, quality, cost)")
print(f"    • Rollback quickly if issues arise")

In [ ]:
# Cost comparison: build vs buy
print("Build vs Buy: Self-Host vs API Provider\n")

print(f"  {'':30} {'Self-Host':>15} {'API (Claude/GPT)':>18}")
print(f"  {'─'*30} {'─'*15} {'─'*18}")
print(f"  {'Upfront cost':<30} {'$10K-100K':>15} {'$0':>18}")
print(f"  {'Per-query cost (1K tokens)':<30} {'~$0.001':>15} {'~$0.003-0.06':>18}")
print(f"  {'Latency control':<30} {'Full':>15} {'Limited':>18}")
print(f"  {'Data privacy':<30} {'Full (on-prem)':>15} {'Shared servers':>18}")
print(f"  {'Model quality':<30} {'Good (7-70B)':>15} {'Best (frontier)':>18}")
print(f"  {'Maintenance':<30} {'You do it':>15} {'Provider does':>18}")
print(f"  {'Scaling':<30} {'You manage':>15} {'Automatic':>18}")
print(f"  {'Time to production':<30} {'Weeks-months':>15} {'Hours-days':>18}")

print(f"\n  When to self-host:")
print(f"    • Very high volume (millions of requests/day)")
print(f"    • Strict privacy requirements (healthcare, defense)")
print(f"    • Need custom model (fine-tuned, domain-specific)")
print(f"    • Latency-critical applications")

print(f"\n  When to use an API:")
print(f"    • Starting out (fast iteration, no infra)")
print(f"    • Need frontier quality (GPT-4, Claude)")
print(f"    • Variable/unpredictable traffic")
print(f"    • Small team (can't maintain GPU fleet)")

print(f"\n  Many companies: API for prototyping → self-host when proven")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **Model serving** | HTTP API pattern — model on GPU, clients send requests |
| **Optimization** | Quantization, KV-cache, batching, speculative decoding |
| **Scaling** | Horizontal (more GPUs) + autoscaling + caching |
| **Monitoring** | Track latency, errors, quality, cost continuously |
| **A/B testing** | Compare model versions on real traffic before rollout |
| **Model drift** | Data/concept/label drift — retrain when detected |
| **Safety** | Input/output filters, rate limiting, logging |
| **MLOps** | Full lifecycle: data → train → deploy → monitor → retrain |
| **Build vs buy** | Self-host for scale/privacy, API for speed/quality |

---

## 🎉 Book Complete!

You now understand the full journey from raw data to a deployed AI system:

```
Part 1: ML Foundations     — how machines learn from data
Part 2: Neural Networks    — deep learning, backpropagation
Part 3: Generative AI      — how LLMs are built inside the black box
Part 4: Applied AI         — making it real (data, eval, RAG, deploy)
```

The black box is no longer black.